In [1]:
import networkx as nx
import json
from networkx.readwrite import json_graph

In [2]:
def parse_rfa(file_path):
    with open(file_path, 'rt', encoding='utf-8') as f:
        content = f.read()

    entries = content.strip().split("\n\n")
    data = []

    for entry in entries:
        record = {}
        for line in entry.split("\n"):
            if ":" in line:
                key, value = line.split(":", 1)
                record[key.strip()] = value.strip()
        data.append(record)

    return data

In [3]:
def build_graph(data):
    G = nx.MultiDiGraph()

    for d in data:
        src = d.get("SRC", "").strip()
        tgt = d.get("TGT", "").strip()

        if not src or not tgt:
            continue  # skip malformed records

        G.add_edge(
            src,
            tgt,
            vote=int(d["VOT"]),
            text=d["TXT"],
            result=int(d["RES"]),
            year=int(d["YEA"]),
            date=d["DAT"]
        )

    return G

In [4]:
data = parse_rfa("wiki-RfA.txt")
G = build_graph(data)

print(f"{G.number_of_nodes()} nodes")
print(f"{G.number_of_edges()} edges")

11377 nodes
196614 edges


In [5]:
data = json_graph.node_link_data(G)

with open("rfa_graph.json", "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False)